In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

# ==========================
# Load Dataset
# ==========================
df = pd.read_csv("dataset.csv")

# Column containing missing values
target_col = "Salary"

# Choose model: "rf" for Random Forest, "dt" for Decision Tree
model_type = "rf"

# ==========================
# Split Data
# ==========================
known = df[df[target_col].notna()].copy()
missing = df[df[target_col].isna()].copy()

if missing.empty:
    print(f"No missing values found in '{target_col}'")

else:

    # Features and Target
    X_train = known.drop(columns=[target_col])
    y_train = known[target_col]

    X_missing = missing.drop(columns=[target_col])

    # Convert categorical variables
    X_train = pd.get_dummies(X_train, drop_first=True)
    X_missing = pd.get_dummies(X_missing, drop_first=True)

    # Match columns
    X_missing = X_missing.reindex(
        columns=X_train.columns,
        fill_value=0
    )

    # ==========================
    # Model Selection
    # ==========================
    if model_type == "dt":

        model = DecisionTreeRegressor(
            max_depth=5,
            random_state=42
        )

    else:

        model = RandomForestRegressor(
            n_estimators=200,
            max_depth=10,
            random_state=42,
            n_jobs=-1
        )

    # Train Model
    model.fit(X_train, y_train)

    # Predict Missing Salaries
    predicted_salary = model.predict(X_missing)

    # Fill Missing Values
    df.loc[df[target_col].isna(), target_col] = predicted_salary

    print("Missing Salary values have been imputed.")

# ==========================
# Save Dataset
# ==========================
df.to_csv("employee_data_imputed.csv", index=False)

print("Imputed dataset saved as employee_data_imputed.csv")

# Display updated data
print(df.head())

Missing Salary values have been imputed.
Imputed dataset saved as employee_data_imputed.csv
    Age  Experience Education  Department        Salary
0  25.0         2.0  Bachelor          IT  35000.000000
1  30.0         5.0    Master          HR  50000.000000
2  28.0         3.0  Bachelor          IT  36668.333333
3  35.0         8.0    Master     Finance  70000.000000
4  40.0        12.0       PhD  Management  90000.000000


In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor, XGBClassifier
from sklearn.preprocessing import LabelEncoder

# ==========================
# Load Dataset
# ==========================
df = pd.read_csv("dataset.csv")

# ==========================
# Function to impute one column
# ==========================
def impute_column(df, target_col):

    known = df[df[target_col].notna()].copy()
    missing = df[df[target_col].isna()].copy()

    if missing.empty:
        print(f"{target_col}: No missing values")
        return df

    # Split features/target
    X_train = known.drop(columns=[target_col])
    X_missing = missing.drop(columns=[target_col])

    # One-hot encode features
    X_train = pd.get_dummies(X_train, drop_first=True)
    X_missing = pd.get_dummies(X_missing, drop_first=True)

    # Align columns
    X_missing = X_missing.reindex(columns=X_train.columns, fill_value=0)

    # ==========================
    # Numeric target → Regression
    # ==========================
    if pd.api.types.is_numeric_dtype(known[target_col]):

        y_train = known[target_col]

        model = XGBRegressor(
            n_estimators=200,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        )

        model.fit(X_train, y_train)
        predictions = model.predict(X_missing)

    # ==========================
    # Categorical target → Classification
    # ==========================
    else:

        encoder = LabelEncoder()
        y_train = encoder.fit_transform(known[target_col].astype(str))

        model = XGBClassifier(
            n_estimators=200,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            eval_metric="mlogloss"
        )

        model.fit(X_train, y_train)
        predictions = encoder.inverse_transform(model.predict(X_missing))

    # Fill missing values
    df.loc[df[target_col].isna(), target_col] = predictions

    print(f"{target_col}: Missing values imputed")

    return df


# ==========================
# Impute ALL columns
# ==========================
for col in df.columns:
    if df[col].isna().sum() > 0:
        df = impute_column(df, col)

# ==========================
# Save Final Dataset
# ==========================
df.to_csv("dataset_imputed_all_xgboost.csv", index=False)

print("All missing values handled using XGBoost.")
print(df)

Age: Missing values imputed
Experience: Missing values imputed
Department: Missing values imputed
Salary: Missing values imputed
All missing values handled using XGBoost.
          Age  Experience Education  Department         Salary
0   25.000000    2.000000  Bachelor          IT   35000.000000
1   30.000000    5.000000    Master          HR   50000.000000
2   28.000000    3.000000  Bachelor          IT   39496.328125
3   35.000000    8.000000    Master     Finance   70000.000000
4   40.000000   12.000000       PhD  Management   90000.000000
5   27.030853    2.000000  Bachelor          IT   34065.742188
6   32.528992    6.000000    Master     Finance   60000.000000
7   45.000000   15.000000       PhD  Management  110000.000000
8   29.000000    4.000000  Bachelor          HR   42911.195312
9   38.000000   10.000000    Master     Finance   80000.000000
10  26.000000    1.000000  Bachelor          IT   32000.000000
11  31.000000    5.343384    Master          HR   50582.800781
12  36.000

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# ==========================
# Load Dataset
# ==========================
df = pd.read_csv("dataset.csv")

# ==========================
# Function to impute one column
# ==========================
def impute_column(df, target_col):

    known = df[df[target_col].notna()].copy()
    missing = df[df[target_col].isna()].copy()

    if missing.empty:
        print(f"{target_col}: No missing values")
        return df

    # Split features/target
    X_train = known.drop(columns=[target_col])
    X_missing = missing.drop(columns=[target_col])

    # One-hot encode categorical features
    X_train = pd.get_dummies(X_train, drop_first=True)
    X_missing = pd.get_dummies(X_missing, drop_first=True)

    # Align columns
    X_missing = X_missing.reindex(columns=X_train.columns, fill_value=0)

    # ==========================
    # Numeric target → Regression
    # ==========================
    if pd.api.types.is_numeric_dtype(known[target_col]):

        y_train = known[target_col]

        model = RandomForestRegressor(
            n_estimators=200,
            max_depth=10,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train, y_train)
        predictions = model.predict(X_missing)

    # ==========================
    # Categorical target → Classification
    # ==========================
    else:

        encoder = LabelEncoder()
        y_train = encoder.fit_transform(known[target_col].astype(str))

        model = RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train, y_train)
        predictions = encoder.inverse_transform(model.predict(X_missing))

    # Fill missing values
    df.loc[df[target_col].isna(), target_col] = predictions

    print(f"{target_col}: Missing values imputed using Random Forest")

    return df


# ==========================
# Impute ALL columns
# ==========================
for col in df.columns:
    if df[col].isna().sum() > 0:
        df = impute_column(df, col)

# ==========================
# Save Final Dataset
# ==========================
df.to_csv("dataset_imputed_all_randomforest.csv", index=False)

print("All missing values handled using Random Forest.")
print(df)

Age: Missing values imputed using Random Forest
Experience: Missing values imputed using Random Forest
Department: Missing values imputed using Random Forest
Salary: Missing values imputed using Random Forest
All missing values handled using Random Forest.
          Age  Experience Education  Department    Salary
0   25.000000       2.000  Bachelor          IT   35000.0
1   30.000000       5.000    Master          HR   50000.0
2   28.000000       3.000  Bachelor          IT   37175.0
3   35.000000       8.000    Master     Finance   70000.0
4   40.000000      12.000       PhD  Management   90000.0
5   27.301833       2.000  Bachelor          IT   35835.0
6   32.448083       6.000    Master     Finance   60000.0
7   45.000000      15.000       PhD  Management  110000.0
8   29.000000       4.000  Bachelor          HR   45565.0
9   38.000000      10.000    Master     Finance   80000.0
10  26.000000       1.000  Bachelor          IT   32000.0
11  31.000000       5.095    Master          HR

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder

# ==========================
# Load Dataset
# ==========================
df = pd.read_csv("dataset.csv")

# ==========================
# CLEAN DATA (IMPORTANT)
# ==========================
df.replace([" ", "", "NA", "N/A", "null"], np.nan, inplace=True)
df.columns = df.columns.str.strip()

# Convert numeric columns safely
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="ignore")


# ==========================
# Function to impute one column
# ==========================
def impute_column(df, target_col):

    known = df[df[target_col].notna()].copy()
    missing = df[df[target_col].isna()].copy()

    if missing.empty:
        print(f"{target_col}: No missing values")
        return df

    # Split features/target
    X_train = known.drop(columns=[target_col])
    X_missing = missing.drop(columns=[target_col])

    # One-hot encode categorical features
    X_train = pd.get_dummies(X_train, drop_first=True)
    X_missing = pd.get_dummies(X_missing, drop_first=True)

    # Align columns
    X_missing = X_missing.reindex(columns=X_train.columns, fill_value=0)

    # ==========================
    # KNN IMPUTATION (CORE CHANGE)
    # ==========================
    imputer = KNNImputer(n_neighbors=5)

    # Fit on known + missing features together
    full_X = pd.concat([X_train, X_missing])
    full_X_imputed = imputer.fit_transform(full_X)

    # Split back
    X_train_imputed = full_X_imputed[:len(X_train)]
    X_missing_imputed = full_X_imputed[len(X_train):]

    # ==========================
    # Numeric target → average of neighbors
    # ==========================
    if pd.api.types.is_numeric_dtype(known[target_col]):

        y_train = known[target_col].values

        # simple weighted average idea using nearest rows
        from sklearn.neighbors import KNeighborsRegressor

        model = KNeighborsRegressor(n_neighbors=5)
        model.fit(X_train_imputed, y_train)

        predictions = model.predict(X_missing_imputed)

    # ==========================
    # Categorical target → majority vote
    # ==========================
    else:

        encoder = LabelEncoder()
        y_train = encoder.fit_transform(known[target_col].astype(str))

        from sklearn.neighbors import KNeighborsClassifier

        model = KNeighborsClassifier(n_neighbors=5)
        model.fit(X_train_imputed, y_train)

        predictions = encoder.inverse_transform(model.predict(X_missing_imputed))

    # Fill missing values
    df.loc[df[target_col].isna(), target_col] = predictions

    print(f"{target_col}: Missing values imputed using KNN")

    return df


# ==========================
# Impute ALL columns
# ==========================
for col in df.columns:
    if df[col].isna().sum() > 0:
        df = impute_column(df, col)

print("All missing values handled using KNN.")
print(df)

Age: Missing values imputed using KNN
Experience: Missing values imputed using KNN
Department: Missing values imputed using KNN
Salary: Missing values imputed using KNN
All missing values handled using KNN.
     Age  Experience Education  Department    Salary
0   25.0         2.0  Bachelor          IT   35000.0
1   30.0         5.0    Master          HR   50000.0
2   28.0         3.0  Bachelor          IT   38200.0
3   35.0         8.0    Master     Finance   70000.0
4   40.0        12.0       PhD  Management   90000.0
5   26.8         2.0  Bachelor          IT   34600.0
6   32.4         6.0    Master     Finance   60000.0
7   45.0        15.0       PhD  Management  110000.0
8   29.0         4.0  Bachelor          HR   46800.0
9   38.0        10.0    Master     Finance   80000.0
10  26.0         1.0  Bachelor          IT   32000.0
11  31.0         5.0    Master          HR   53000.0
12  36.0         9.0    Master     Finance   75000.0
13  42.0        14.2       PhD  Management  103600.

/tmp/ipykernel_8795/773795947.py:19: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
